# Value at Risk (VaR)

Imagine que você tem **R$ 100.000** investidos na bolsa de valores. Qual é o pior cenário possível - *a perda monetária máxima* - que você aceita tolerar amanhã com 95% de certeza?

In [35]:
import yfinance as yf
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import datetime

In [36]:
dinheiro_investido = 100000.0
confianca = 0.05
anos = 2

In [37]:
data_final = datetime.datetime.now()
data_inicial = data_final - datetime.timedelta(days=anos * 365)
dados = yf.download("PETR4.SA", start=data_inicial, end=data_final, progress=False)
precos = dados['Close'].dropna()
display(precos)

Ticker,PETR4.SA
Date,
2024-04-18,29.200985
2024-04-19,29.699272
2024-04-22,30.410063
2024-04-23,30.351440
2024-04-24,30.212217
...,...
2026-04-13,49.779999
2026-04-14,47.880001
2026-04-15,46.889999


In [42]:
retornos = np.log(precos / precos.shift(1)).dropna()
media = retornos.mean().item()
desvio_padrao = retornos.std().item()

print(f"Média dos retornos: {media}")
print(f"DP dos retornos: {desvio_padrao}")

Média dos retornos: 0.0009202609042518401
DP dos retornos: 0.015421523230065983


In [51]:
# A tabela estática da curva normal diz que a extremidade de 5% equivale à quebra no número de Z-Score -1.64
z = stats.norm.ppf(confianca)

# Fórmula Padrão Global: Valor_da_Carteira * (-Z * Volatilidade_da_Ação - Media_da_Ação)
var_parametrico = dinheiro_investido * (-z * desvio_padrao - media)

print(f"VaR Paramétrico:", var_parametrico)

VaR Paramétrico: 2444.5887513838584


In [52]:
# Organiza todos os retornos do pior pro melhor (-5%, - 4%, -2% .... +1%, +5%)
retornos_limpos = retornos.values.flatten()
retornos_ordenados = np.sort(retornos_limpos)

# Encontra a posição exata (índice) da quebra de 5%
linha_de_corte = int(confianca * len(retornos_ordenados))

# Multiplica pelo seu dinheiro na carteira 
var_historico = -retornos_ordenados[linha_de_corte] * dinheiro_investido

print("VaR Histórico:", var_historico)

VaR Histórico: 2432.138901059842


In [53]:
np.random.seed(42)

# A. Rolando 50 mil dias de bolsa acreditando numa linha perfeitamente ideal (Distribuição Normal)
realidades_normais = np.random.normal(media, desvio_padrao, 50000)
corte_simulacoes = int(confianca * 50000)

realidades_normais_ordenadas = np.sort(realidades_normais)
var_montecarlo_normal = -realidades_normais_ordenadas[corte_simulacoes] * dinheiro_investido

print("VaR Monte Carlo (Normal):", var_montecarlo_normal)

# B. CAUDAS GORDAS (Tragédias Constantes)
# Distribuições como a T-Student sabem que "Cisnes Negros" quebram a matemática perfeita.
realidades_tragedias = stats.t.rvs(df=4, loc=media, scale=desvio_padrao, size=50000)

realidades_tragicas_ordenadas = np.sort(realidades_tragedias)
var_montecarlo_tstudent = -realidades_tragicas_ordenadas[corte_simulacoes] * dinheiro_investido

print("VaR Monte Carlo (Cauda Gorda):", var_montecarlo_tstudent)

VaR Monte Carlo (Normal): 2432.5118207294054
VaR Monte Carlo (Cauda Gorda): 3238.229436632192


In [54]:
tabela_final = pd.DataFrame({
    'Modelo Explicativo da PETR4': ['Paramétrico', 'Histórico', 'Monte Carlo Normal', 'Monte Carlo (Cauda Gorda)'],
    '1 Dia de Risco Máx': [var_parametrico, var_historico, var_montecarlo_normal, var_montecarlo_tstudent],
    '10 Dias de Risco Máx': [var_parametrico*np.sqrt(10), var_historico*np.sqrt(10), var_montecarlo_normal*np.sqrt(10), var_montecarlo_tstudent*np.sqrt(10)]
})
display(tabela_final)

,Modelo Explicativo da PETR4,1 Dia de Risco Máx,10 Dias de Risco Máx
0,Paramétrico,2444.588751,7730.468397
1,Histórico,2432.138901,7691.098513
2,Monte Carlo Normal,2432.511821,7692.277789
3,Monte Carlo (Cauda Gorda),3238.229437,10240.180606
